# RDD

### Import données

In [372]:
import importlib
import credentials
importlib.reload(credentials)
import os 
import pandas as pd
from BDTopo_fonctions import load_gpkg

#vote
remote_path="Elections/fichier_vote_avec_code_postal_clean.csv"
local_path = f"/tmp/{os.path.basename(remote_path)}"
credentials.s3.download_file("mgarbe", remote_path, local_path)
df_vote=pd.read_csv(local_path, sep=",")

#code nuances politiques
remote_path="Elections/nuances.csv"
local_path = f"/tmp/{os.path.basename(remote_path)}"
credentials.s3.download_file("mgarbe", remote_path, local_path)
nuances=pd.read_csv(local_path, sep=",")

#données geo
gdf=load_gpkg("Sitadel/df_clustering_fulldep_1000m3.gpkg") #SANS CLUSTER !!!
gdf=gdf[gdf["Base"]=="Sitadel"].copy()

Téléchargement depuis mgarbe/Sitadel/df_clustering_fulldep_1000m3.gpkg ...
Chargement réussi (324979 lignes)


In [373]:
#on réduit aux communes dans la fenêtre du RDD
seuil=4
df_vote_seuil=df_vote[df_vote["voix_pct"]<50+seuil].copy()

In [374]:
df_vote

,annee,ident_election_ville,Nuance_muni,voix_pct,Nuance_interco
0,2014,01004,LDVD,61.309524,NC
1,2014,01005,LDIV,100.000000,LDVD
2,2014,01007,LDVG,46.779141,NC
3,2014,01010,LDVD,56.153846,NC
4,2014,01014,LUMP,100.000000,LDVD
...,...,...,...,...,...
12258,2020,95607,LDVD,58.329564,LDVD
12259,2020,95612,LDVD,48.525214,LDVD
12260,2020,95637,LDVG,51.761583,LDVG
12261,2020,95652,LDVD,73.375796,NC


In [375]:
id_communes=df_vote_seuil.groupby("annee")["ident_election_ville"].unique()

In [376]:
id_communes

annee
2014    [01007, 01034, 01053, 01093, 01143, 01159, 011...
2020    [01004, 01071, 01160, 01185, 01194, 01249, 012...
Name: ident_election_ville, dtype: object

In [377]:
import geopandas as gpd

# Fonction pour créer la colonne mandat_elec
def assign_mandat(annee):
    if 2014 <= annee <= 2019:
        return 2014  # groupe 1
    elif 2020 <= annee <= 2025:
        return 2020  # groupe 2
    else:
        return None   # pour filtrer les années hors des plages

# Création de la colonne
gdf['mandat_elec'] = gdf['Annee_REF'].apply(assign_mandat)

# Filtrage uniquement sur les périodes souhaitées
gdf = gdf[gdf['mandat_elec'].notna()].copy()

In [378]:
# Normaliser COMM à 5 caractères
gdf['COMM'] = gdf['COMM'].astype(str)
gdf.loc[gdf['COMM'].str.len() == 4, 'COMM'] = '0' + gdf.loc[gdf['COMM'].str.len() == 4, 'COMM']

# Filtrer par mandat_elec et id_communes correspondantes
gdf_RDD_list = []

for mandat in id_communes.index:
    communes = [str(c).zfill(5) for c in id_communes[mandat]]  # s'assure que 5 chiffres
    gdf_RDD_list.append(
        gdf[(gdf['mandat_elec'] == mandat) & (gdf['COMM'].isin(communes))]
    )

# Concaténer les deux groupes
gdf_RDD = pd.concat(gdf_RDD_list, ignore_index=True)

In [379]:
gdf_RDD.groupby("mandat_elec").size()

mandat_elec
2014    6886
2020    3105
dtype: int64

In [380]:
gdf_RDD['COMM'] = gdf_RDD['COMM'].astype(str)
df_vote['ident_election_ville'] = df_vote['ident_election_ville'].astype(str)

# Merge
gdf_RDD = gdf_RDD.merge(
    df_vote[['ident_election_ville', 'Nuance_muni', 'Nuance_interco']],
    left_on='COMM',
    right_on='ident_election_ville',
    how='left'
)

# Optionnel : supprimer la colonne ident_election_ville redondante
gdf_RDD = gdf_RDD.drop(columns=['ident_election_ville']).copy()

In [381]:
nuances

,Nuance,Libellé,Code
0,EXG,Extrême gauche,-1
1,COM,Communiste,-1
2,PG,Parti de Gauche,-1
3,SOC,Socialiste,-1
4,RDG,Radical de Gauche,-1
...,...,...,...
81,LMMD,Liste Majorité-MoDem,1
82,LCEN,Liste d'union du centre,1
83,LDLR,Liste Debout la République,1
84,DLR,Debout la République,1


In [382]:
nuance_dict = pd.Series(nuances['Code'].values, index=nuances['Nuance']).to_dict()
gdf_RDD['Nuance_muni'] = gdf_RDD['Nuance_muni'].map(nuance_dict)
gdf_RDD['Nuance_interco'] = gdf_RDD['Nuance_interco'].map(nuance_dict)
gdf_RDD=gdf_RDD[gdf_RDD["Nuance_muni"]!=0].copy()

## RESULTATS 

In [383]:
gdf_RDD.groupby(['mandat_elec','Nuance_muni'])['COMM'].nunique().reset_index(name='nb_communes')

,mandat_elec,Nuance_muni,nb_communes
0,2014,-1,743
1,2014,1,1240
2,2020,-1,341
3,2020,1,495


Nb permis

In [384]:
obs_par_commune = gdf_RDD.groupby(['Nuance_muni','mandat_elec','COMM']).size().reset_index(name='nb_obs')
RDD_nb_bati = obs_par_commune.groupby(['mandat_elec','Nuance_muni'])['nb_obs'].mean()
RDD_nb_bati=RDD_nb_bati.reset_index()
print(RDD_nb_bati)


   mandat_elec  Nuance_muni    nb_obs
0         2014           -1  5.806191
1         2014            1  5.803226
2         2020           -1  6.747801
3         2020            1  6.846465


In [385]:
mandats = RDD_nb_bati['mandat_elec'].unique()

for mandat in mandats:
    # Sélection des communes pour ce mandat
    df_m = gdf_RDD[gdf_RDD['mandat_elec'] == mandat]
    
    # Extraire SURF_CREEE par groupe Nuance_muni
    groupe_neg1 = df_m[df_m['Nuance_muni'] == -1].groupby('COMM').size()
    groupe_pos1 = df_m[df_m['Nuance_muni'] == 1].groupby('COMM').size()
    
    # Test t de Welch (variances inégales)
    t_stat, p_val = stats.ttest_ind(groupe_neg1, groupe_pos1, equal_var=False)
    
    print(f"Mandat {mandat}: t = {t_stat:.3f}, p = {p_val:.4f}")
    if p_val < 0.1:
        print("→ Différence significative de surface moyenne entre les groupes")
    else:
        print("→ Pas de différence significative de surface moyenne entre les groupes")

Mandat 2014: t = 0.008, p = 0.9935
→ Pas de différence significative de surface moyenne entre les groupes
Mandat 2020: t = -0.170, p = 0.8652
→ Pas de différence significative de surface moyenne entre les groupes


Surface créee

In [386]:
surf_par_commune = gdf_RDD.groupby(['Nuance_muni', 'mandat_elec', 'COMM'])['SURF_CREEE'].sum().reset_index(name='surf_commune')
RDD_surf = surf_par_commune.groupby(['mandat_elec','Nuance_muni'])['surf_commune'].mean().reset_index()

print(RDD_surf)

   mandat_elec  Nuance_muni  surf_commune
0         2014           -1  27559.235532
1         2014            1  23393.513710
2         2020           -1  32851.105572
3         2020            1  28814.397980


In [387]:
mandats = RDD_surf['mandat_elec'].unique()

for mandat in mandats:
    # Sélection des communes pour ce mandat
    df_m = gdf_RDD[gdf_RDD['mandat_elec'] == mandat]
    
    # Extraire SURF_CREEE par groupe Nuance_muni
    groupe_neg1 = df_m[df_m['Nuance_muni'] == -1].groupby('COMM')['SURF_CREEE'].sum()
    groupe_pos1 = df_m[df_m['Nuance_muni'] == 1].groupby('COMM')['SURF_CREEE'].sum()
    
    # Test t VOIR WELCH VS STUDENT
    t_stat, p_val = stats.ttest_ind(groupe_neg1, groupe_pos1, equal_var=True) #student
    
    print(f"Mandat {mandat}: t = {t_stat:.3f}, p = {p_val:.4f}")
    if p_val < 0.1:
        print("→ Différence significative de surface moyenne entre les groupes")
    else:
        print("→ Pas de différence significative de surface moyenne entre les groupes")

Mandat 2014: t = 1.797, p = 0.0726
→ Différence significative de surface moyenne entre les groupes
Mandat 2020: t = 0.974, p = 0.3305
→ Pas de différence significative de surface moyenne entre les groupes


Surface moyenne par bat

In [388]:
surf_moyenne = gdf_RDD.groupby(['mandat_elec','Nuance_muni'])['SURF_CREEE'].mean().reset_index(name='surf_moy')
print(surf_moyenne)

   mandat_elec  Nuance_muni     surf_moy
0         2014           -1  4746.525730
1         2014            1  4031.122429
2         2020           -1  4868.416775
3         2020            1  4208.653585


In [389]:
mandats = RDD_surf['mandat_elec'].unique()

for mandat in mandats:
    # Sélection des communes pour ce mandat
    df_m = gdf_RDD[gdf_RDD['mandat_elec'] == mandat]
    
    # Extraire SURF_CREEE par groupe Nuance_muni
    groupe_neg1 = df_m[df_m['Nuance_muni'] == -1]['SURF_CREEE']
    groupe_pos1 = df_m[df_m['Nuance_muni'] == 1]['SURF_CREEE']
    
    # Test t de Welch (variances inégales)
    t_stat, p_val = stats.ttest_ind(groupe_neg1, groupe_pos1, equal_var=False)
    
    print(f"Mandat {mandat}: t = {t_stat:.3f}, p = {p_val:.4f}")
    if p_val < 0.1:
        print("→ Différence significative de surface moyenne entre les groupes")
    else:
        print("→ Pas de différence significative de surface moyenne entre les groupes")

Mandat 2014: t = 4.461, p = 0.0000
→ Différence significative de surface moyenne entre les groupes
Mandat 2020: t = 2.061, p = 0.0394
→ Différence significative de surface moyenne entre les groupes
